# Playground - Chip Layout & Measurement Planner

This notebook provides a testing environment for processing and visualizing information from a **photonic chip** described in an Excel file (`chip_layout_normalized.xlsx`).

## Purpose

- Load and organize the chip data (devices, ports, and measurement configurations).
- Build a measurement plan based on the enabled configurations.
- Simulate optical fiber positioning for each measurement.

## Excel File Structure

The Excel file contains four worksheets:

- **Metadata**: General information about the chip.
- **Devices**: List of photonic devices (e.g., MMI, MZI, BTS).
- **Ports**: Coordinates (X, Y) of each port for every device.
- **Measurements**: Input/output port combinations to be measured, including an enable/disable flag.

## Workflow

1. Read the Excel worksheets using `pandas`.
2. Build a nested `chip` dictionary containing devices and their corresponding ports.
3. Generate a `measurement_plan` list containing all enabled measurements.
4. Simulate optical fiber movements for each active measurement.

## Usage

This notebook serves as a starting point for developing automated measurement routines for photonic integrated circuits (PICs). It can also be used to validate chip layouts and verify measurement configurations before executing experiments on the Suruga-Seiki alignment platform.

In [25]:
import pandas as pd
from pprint import pprint

# ======================================================
# Configuración
# ======================================================

EXCEL_FILE = "chip_layout_normalized.xlsx"

El archivo excel se compone de 4 hojas Metadata, Devices, Ports, Measurements

In [26]:
# ======================================================
# Leer el Excel
# ======================================================

metadata_df = pd.read_excel(
    EXCEL_FILE,
    sheet_name="Metadata"
)

devices_df = pd.read_excel(
    EXCEL_FILE,
    sheet_name="Devices"
)

ports_df = pd.read_excel(
    EXCEL_FILE,
    sheet_name="Ports"
)

measurements_df = pd.read_excel(
    EXCEL_FILE,
    sheet_name="Measurements"
)

Iterrows entrega el índice de la fila y la fila completa, este es un ejemplo de lo que sucedería si se usasen ambas variables

In [40]:
for indice, row in devices_df.iterrows():
    print(indice, row["Device"], row["Type"], row["Description"], row["Measure"])

0 BTS_01 BTS1x16 Binary Tree Splitter NO
1 MMI_01 MMI1x2 1x2 splitter NO
2 MMI_02 MMI2X2 2x2 splitter YES
3 MZI_01 MZI Reference Mach-Zehnder NO


Sin embargo solo me intereza lo que ocurre con `row` por eso se usa `_` en vez de `indice`, en python `_` significa que la variable existe pero no nos interesa de momento


In [41]:
# ======================================================
# Construir el objeto CHIP
# ======================================================

chip = {}

# ------------------------------------------------------
# Añadir los dispositivos
# ------------------------------------------------------

for _, row in devices_df.iterrows():

    chip[row["Device"]] = {

        "type": row["Type"],

        "measure": row["Measure"],

        "ports": {} # Es un diccionario vacío que se llenará más adelante con los puertos del dispositivo

    }

In [42]:
# ------------------------------------------------------
# Añadir los puertos a cada dispositivo
# ------------------------------------------------------

for _, row in ports_df.iterrows():
    # Chequea cada una de las filas del DataFrame de puertos y añade la información 
    # correspondiente al diccionario del dispositivo correspondiente en el objeto CHIP.

    device = row["Device"] # Esto es valido debido a que cada puerto tiene un dispositivo 
    #asociado en la columna "Device" que coincide con el nombre del dispositivo en el DataFrame de devices.

    chip[device]["ports"][row["Port"]] = {

        "x": row["X (um)"],

        "y": row["Y (um)"],

        "direction": row["Direction"]

    }

In [43]:
# ======================================================
# Mostrar la estructura cargada
# ======================================================

print("=" * 80)
print("CHIP CARGADO")
print("=" * 80)

pprint(chip)

CHIP CARGADO
{'BTS_01': {'measure': 'NO',
            'ports': {'P1': {'direction': 'IN', 'x': 952.5, 'y': 300},
                      'P10': {'direction': 'OUT', 'x': 1524.0, 'y': 300},
                      'P11': {'direction': 'OUT', 'x': 1587.5, 'y': 300},
                      'P12': {'direction': 'OUT', 'x': 1651.0, 'y': 300},
                      'P13': {'direction': 'OUT', 'x': 1714.5, 'y': 300},
                      'P14': {'direction': 'OUT', 'x': 1778.0, 'y': 300},
                      'P15': {'direction': 'OUT', 'x': 1841.5, 'y': 300},
                      'P16': {'direction': 'OUT', 'x': 1905.0, 'y': 300},
                      'P17': {'direction': 'OUT', 'x': 1968.5, 'y': 300},
                      'P2': {'direction': 'OUT', 'x': 1016.0, 'y': 300},
                      'P3': {'direction': 'OUT', 'x': 1079.5, 'y': 300},
                      'P4': {'direction': 'OUT', 'x': 1143.0, 'y': 300},
                      'P5': {'direction': 'OUT', 'x': 1206.5, 'y': 300},
   

In [44]:
measurement_plan = []

# ======================================================
# Construir la lista de mediciones
# ======================================================
for _, row in measurements_df.iterrows():

    measurement = {

        "device": row["Device"],

        "input": row["Input"],

        "output": row["Output"],

        "enabled": row["Enabled"] == "YES"

        # "status": "PENDING",

        # "power": None,

        # "timestamp": None

    }

    measurement_plan.append(measurement)

In [45]:
# ======================================================
# Mostrar el plan de medidas
# ======================================================

print("=" * 80)
print("PLAN DE MEDIDAS")
print("=" * 80)

pprint(measurement_plan)

PLAN DE MEDIDAS
[{'device': 'BTS_01', 'enabled': False, 'input': 'P1', 'output': 'P2'},
 {'device': 'BTS_01', 'enabled': False, 'input': 'P1', 'output': 'P3'},
 {'device': 'BTS_01', 'enabled': False, 'input': 'P1', 'output': 'P4'},
 {'device': 'BTS_01', 'enabled': False, 'input': 'P1', 'output': 'P5'},
 {'device': 'BTS_01', 'enabled': False, 'input': 'P1', 'output': 'P6'},
 {'device': 'BTS_01', 'enabled': False, 'input': 'P1', 'output': 'P7'},
 {'device': 'BTS_01', 'enabled': False, 'input': 'P1', 'output': 'P8'},
 {'device': 'BTS_01', 'enabled': False, 'input': 'P1', 'output': 'P9'},
 {'device': 'BTS_01', 'enabled': False, 'input': 'P1', 'output': 'P10'},
 {'device': 'BTS_01', 'enabled': False, 'input': 'P1', 'output': 'P11'},
 {'device': 'BTS_01', 'enabled': False, 'input': 'P1', 'output': 'P12'},
 {'device': 'BTS_01', 'enabled': False, 'input': 'P1', 'output': 'P13'},
 {'device': 'BTS_01', 'enabled': False, 'input': 'P1', 'output': 'P14'},
 {'device': 'BTS_01', 'enabled': False, 'in

Deseamos acceder a los elementos que conforman el device `MMI_01` para lo cual podemos hacer uso del siguiente código

In [15]:
device = chip["MMI_01"]

# Acceder a propiedades principales
print(device["type"], "→", type(device["type"]).__name__)      # MMI1x2 → str
print(device["measure"], "→", type(device["measure"]).__name__)   # NO → str

# Acceder a un puerto específico
print(device["ports"]["P1"], "→", type(device["ports"]["P1"]).__name__)        # {'x': 762.0, 'y': 200} → dict
print(device["ports"]["P1"]["x"], "→", type(device["ports"]["P1"]["x"]).__name__)   # 762.0 → float
print(device["ports"]["P1"]["y"], "→", type(device["ports"]["P1"]["y"]).__name__)   # 200 → float

MMI1x2 → str
NO → str
{'x': 762.0, 'y': 200} → dict
762.0 → float
200 → int


In [30]:
for measurement in measurement_plan:

    # ¿Esta medición está habilitada?
    if not measurement["enabled"]:
        continue

    device = chip[measurement["device"]]

    input_port = device["ports"][measurement["input"]]

    output_port = device["ports"][measurement["output"]]

    print("=" * 50)

    print(f"Dispositivo : {measurement['device']}")
    print(f"Entrada     : {measurement['input']}")
    print(f"Salida      : {measurement['output']}")

    print(
        f"Mover fibra izquierda -> "
        f"({input_port['x']}, {input_port['y']})"
    )

    print(
        f"Mover fibra derecha   -> "
        f"({output_port['x']}, {output_port['y']})"
    )

    # --------------------------------------------------
    # Rutinas de posicionamiento y medición
    # --------------------------------------------------

Dispositivo : MMI_02
Entrada     : P1
Salida      : P3
Mover fibra izquierda -> (2635.0, 100)
Mover fibra derecha   -> (2762.0, 100)
Dispositivo : MMI_02
Entrada     : P1
Salida      : P4
Mover fibra izquierda -> (2635.0, 100)
Mover fibra derecha   -> (2825.5, 100)
Dispositivo : MMI_02
Entrada     : P2
Salida      : P3
Mover fibra izquierda -> (2698.5, 100)
Mover fibra derecha   -> (2762.0, 100)
Dispositivo : MMI_02
Entrada     : P2
Salida      : P4
Mover fibra izquierda -> (2698.5, 100)
Mover fibra derecha   -> (2825.5, 100)


### Reflection on the Measurement Plan

At the moment, it has not been considered that the Excel file might not include a predefined measurement plan. Although it would be possible to generate every input/output permutation automatically, this approach could quickly become impractical and produce many unnecessary measurements.

A more sensible solution would be to provide a function that generates the measurement plan and adds it to the Excel file. The user could then simply enable or disable the port combinations they wish to measure, resulting in a clearer, more manageable, and user-friendly workflow.

In [46]:
chip.items()

dict_items([('BTS_01', {'type': 'BTS1x16', 'measure': 'NO', 'ports': {'P1': {'x': 952.5, 'y': 300, 'direction': 'IN'}, 'P2': {'x': 1016.0, 'y': 300, 'direction': 'OUT'}, 'P3': {'x': 1079.5, 'y': 300, 'direction': 'OUT'}, 'P4': {'x': 1143.0, 'y': 300, 'direction': 'OUT'}, 'P5': {'x': 1206.5, 'y': 300, 'direction': 'OUT'}, 'P6': {'x': 1270.0, 'y': 300, 'direction': 'OUT'}, 'P7': {'x': 1333.5, 'y': 300, 'direction': 'OUT'}, 'P8': {'x': 1397.0, 'y': 300, 'direction': 'OUT'}, 'P9': {'x': 1460.5, 'y': 300, 'direction': 'OUT'}, 'P10': {'x': 1524.0, 'y': 300, 'direction': 'OUT'}, 'P11': {'x': 1587.5, 'y': 300, 'direction': 'OUT'}, 'P12': {'x': 1651.0, 'y': 300, 'direction': 'OUT'}, 'P13': {'x': 1714.5, 'y': 300, 'direction': 'OUT'}, 'P14': {'x': 1778.0, 'y': 300, 'direction': 'OUT'}, 'P15': {'x': 1841.5, 'y': 300, 'direction': 'OUT'}, 'P16': {'x': 1905.0, 'y': 300, 'direction': 'OUT'}, 'P17': {'x': 1968.5, 'y': 300, 'direction': 'OUT'}}}), ('MMI_01', {'type': 'MMI1x2', 'measure': 'NO', 'ports'

In [47]:
for device_name, device in chip.items():
    print(f"Nombre del Dispositivo: {device_name}")
    print(f"Informacion: {device}")

Nombre del Dispositivo: BTS_01
Informacion: {'type': 'BTS1x16', 'measure': 'NO', 'ports': {'P1': {'x': 952.5, 'y': 300, 'direction': 'IN'}, 'P2': {'x': 1016.0, 'y': 300, 'direction': 'OUT'}, 'P3': {'x': 1079.5, 'y': 300, 'direction': 'OUT'}, 'P4': {'x': 1143.0, 'y': 300, 'direction': 'OUT'}, 'P5': {'x': 1206.5, 'y': 300, 'direction': 'OUT'}, 'P6': {'x': 1270.0, 'y': 300, 'direction': 'OUT'}, 'P7': {'x': 1333.5, 'y': 300, 'direction': 'OUT'}, 'P8': {'x': 1397.0, 'y': 300, 'direction': 'OUT'}, 'P9': {'x': 1460.5, 'y': 300, 'direction': 'OUT'}, 'P10': {'x': 1524.0, 'y': 300, 'direction': 'OUT'}, 'P11': {'x': 1587.5, 'y': 300, 'direction': 'OUT'}, 'P12': {'x': 1651.0, 'y': 300, 'direction': 'OUT'}, 'P13': {'x': 1714.5, 'y': 300, 'direction': 'OUT'}, 'P14': {'x': 1778.0, 'y': 300, 'direction': 'OUT'}, 'P15': {'x': 1841.5, 'y': 300, 'direction': 'OUT'}, 'P16': {'x': 1905.0, 'y': 300, 'direction': 'OUT'}, 'P17': {'x': 1968.5, 'y': 300, 'direction': 'OUT'}}}
Nombre del Dispositivo: MMI_01
Info

In [48]:
from openpyxl import load_workbook

In [49]:
# ======================================================
# Abrir el archivo Excel
# ======================================================

# Nombre del archivo sobre el que vamos a trabajar.
excel_file = "chip_layout.xlsx"

# Abrimos el archivo de excel, a partir de ahora wb es un objeto que representa 
# el libro de Excel y nos permite acceder a sus hojas y celdas.

wb = load_workbook(excel_file)

# ======================================================
# Crear la hoja Measurements
# ======================================================
# Si la hoja Measurements ya existe, es eliminada para evitar duplicados

if "Measurements" in wb.sheetnames:
    del wb["Measurements"]

# Se crea la hoja measurements ws es un objeto que representa la hoja Measurements 
# y nos permite acceder a sus celdas y filas.
ws = wb.create_sheet("Measurements")

# ======================================================
# Escribir la primera fila (encabezados)
# ======================================================

ws.append([

    "Device",

    "Input",

    "Output",

    "Enabled",

    "Status"

])

In [50]:
# Recorremos todos los dispositivos del chip
for device_name, device in chip.items():

    inputs = []
    outputs = []

    # Recorrer todos los puertos del dispositivo
    for port_name, port in device["ports"].items():

        if port["direction"] == "IN":
            inputs.append(port_name)

        elif port["direction"] == "OUT":
            outputs.append(port_name)

    # Generar todas las combinaciones Entrada-Salida
    for input_port in inputs:

        for output_port in outputs:

            ws.append([

                device_name,

                input_port,

                output_port,

                "YES",

                "Pending"

            ])

In [51]:
# ======================================================
# Guardar
# ======================================================

wb.save(excel_file)

print("Hoja Measurements creada correctamente.")

Hoja Measurements creada correctamente.
